# Выполнение запросов при соединении с БД
## Обрабатываем данные в PostGIS через интерфейс Spark. Выполнение происходит в PostGIS, не на Sedona

# Инициализация

In [1]:
import os
import sys

from sedona.spark import SedonaContext

# Явно указываем путь к Python для воркеров
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Настройка Hadoop
os.environ['HADOOP_HOME'] = r'C:\Hadoop\hadoop-3.3.6'

# Пакеты для Spark 3.5.4 со Scala 2.12
additional_packages = [
    "org.apache.sedona:sedona-spark-3.5_2.12:1.8.0",
    "org.datasyslab:geotools-wrapper:1.8.0-33.1"
]

config = SedonaContext.builder() \
    .appName("SedonaApp") \
    .config("spark.jars.packages", ",".join(additional_packages)) \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator") \
    .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions,org.apache.sedona.viz.sql.SedonaVizExtensions") \
    .config("spark.jars", r"D:\Artem\Work\amtech_projects\postgresql-42.7.13.jar") \
    .master("local[*]") \
    .getOrCreate()

sedona = SedonaContext.create(config)


# Декоратор для замера скорости выполнения запроса

In [2]:
import time
from functools import wraps

def timer_sedona(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"Функция {func.__name__} выполнена за {end - start:.4f} секунд")
        return result
    return wrapper

In [3]:
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
postgresql_url = f"jdbc:postgresql://{credentials.get('host')}:{credentials.get('port')}/gisdb_8411_250226?currentSchema=egip"

# Обработка запроса

In [4]:
@timer_sedona
def execute_query(session, query):
    result_df = session.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({query}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
    result_df.show(10)

In [5]:
def execute_query_all(session, query):
    query_limited = query + "1"
    print("1 запись: ")
    execute_query(session, query_limited)
    
    query_limited = query + "10"
    print("\n 10 записей: ")
    execute_query(session, query_limited)
    
    query_limited = query + "100"
    print("\n 100 записей: ")
    execute_query(session, query_limited)
    
    query_limited = query + "1000"
    print("\n 1K записей: ")
    execute_query(session, query_limited)
    
    query_limited = query + "10000"
    print("\n 10K записей: ")
    execute_query(session, query_limited)
    
    query_limited = query + "100000"
    print("\n 100K записей: ")
    execute_query(session, query_limited)

# Вычисление площади

In [12]:
# запрос
# sql_area = """SELECT id, layer_id, ST_Area(ST_Transform(ST_SetSRID(geometry, 4326), 'EPSG:3857')) as area FROM public.features_plain LIMIT """
sql_area = """ SELECT id, layer_id, geometry FROM public.features_plain LIMIT 100000 """
result_df = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({sql_area}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
result_df.createOrReplaceTempView("spatial_table")
sql_area = """SELECT id, layer_id, ST_Area(ST_Transform(ST_SetSRID(ST_GeomFromWKB(geometry), 4326), 'EPSG:3857')) FROM spatial_table"""
new_df = sedona.sql(sql_area)
new_df.show(10)

+-------+--------+----------------------------------------------------------------------------+
|     id|layer_id|st_area(st_transform(st_setsrid(st_geomfromwkb(geometry), 4326), EPSG:3857))|
+-------+--------+----------------------------------------------------------------------------+
|3442185|     164|                                                                         0.0|
|3442186|     164|                                                                         0.0|
|3442187|     164|                                                                         0.0|
|3442188|     164|                                                                         0.0|
|3442189|     164|                                                                         0.0|
|3442190|     164|                                                                         0.0|
|3442191|     164|                                                                         0.0|
|3442192|     164|                      

# Вычисление геометрия полигона в WKT формате

In [6]:
# запрос
sql_area = """SELECT id, layer_id, ST_AsText(geometry) as geom_wkt FROM public.features_plain LIMIT """
execute_query_all(sedona, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id|            geom_wkt|
+-------+--------+--------------------+
|3700902|     170|MULTIPOLYGON(((37...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.7323 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id|            geom_wkt|
+-------+--------+--------------------+
|3700902|     170|MULTIPOLYGON(((37...|
|3700903|     170|MULTIPOLYGON(((37...|
|3700904|     170|MULTIPOLYGON(((37...|
|3700905|     170|MULTIPOLYGON(((37...|
|3700906|     170|MULTIPOLYGON(((37...|
|3700907|     170|MULTIPOLYGON(((37...|
|3700908|     170|MULTIPOLYGON(((37...|
|3700917|     170|MULTIPOLYGON(((37...|
|3700919|     170|MULTIPOLYGON(((37...|
|3591444|     169|POINT(37.53410073...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5837 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id|            geom_wkt|
+-------+--------+

# Центроид полигона в WKT формате

In [7]:
sql_area = """SELECT id, layer_id, ST_AsText(ST_Centroid(geometry)) as centroid_wkt FROM public.features_plain LIMIT """
execute_query_all(sedona, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id|        centroid_wkt|
+-------+--------+--------------------+
|3812505|     173|POINT(37.21068029...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5155 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id|        centroid_wkt|
+-------+--------+--------------------+
|3812505|     173|POINT(37.21068029...|
|3812639|     173|POINT(36.95436746...|
|3812510|     173|POINT(37.29918235...|
|3812511|     173|POINT(37.55914812...|
|3812512|     173|POINT(37.54444113...|
|3812513|     173|POINT(37.05284200...|
|3405737|     164|POINT(37.55211181...|
|3405738|     164|POINT(37.56230022...|
|3812514|     173|POINT(37.60156161...|
|3812515|     173|POINT(37.31102756...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5035 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id|        centroid_wkt|
+-------+--------+

# Значение периметра полигона

In [10]:
sql_area = """SELECT id, layer_id, ST_Perimeter(ST_Transform(ST_SetSRID(geometry, 4326), 'EPSG:3857'))
 as perimeter FROM public.features_plain LIMIT """
execute_query_all(sedona, sql_area)

1 запись: 
+-------+--------+------------------+
|     id|layer_id|         perimeter|
+-------+--------+------------------+
|3531860|     166|1455.7507552908507|
+-------+--------+------------------+

Функция execute_query выполнена за 0.5523 секунд

 10 записей: 
+-------+--------+------------------+
|     id|layer_id|         perimeter|
+-------+--------+------------------+
|3531860|     166|1455.7507552908507|
|3531861|     166| 791.5839463761625|
|3612999|     169|               0.0|
|3613000|     169|               0.0|
|3613392|     169|               0.0|
|3531862|     166|  1791.13055789384|
|3531863|     166|1791.1361014907795|
|3531864|     166|1448.7255603837639|
|3391521|     163|               0.0|
|3531866|     166|1827.2555383931144|
+-------+--------+------------------+

Функция execute_query выполнена за 0.5175 секунд

 100 записей: 
+-------+--------+------------------+
|     id|layer_id|         perimeter|
+-------+--------+------------------+
|3531860|     166|1455

# Координаты центроида полигона

In [18]:
sql_area = """
WITH calc_centroid AS (
    SELECT 
        id,
        layer_id,
        ST_Centroid(geometry) AS centroid 
    FROM public.features_plain 
    LIMIT 1
)
SELECT 
    id,
    layer_id,
    ST_X(centroid) AS X_centroid,
    ST_Y(centroid) AS Y_centroid
FROM calc_centroid
"""
execute_query(sedona, sql_area)

sql_area = """
WITH calc_centroid AS (
    SELECT 
        id,
        layer_id,
        ST_Centroid(geometry) AS centroid 
    FROM public.features_plain 
    LIMIT 100000
)
SELECT 
    id,
    layer_id,
    ST_X(centroid) AS X_centroid,
    ST_Y(centroid) AS Y_centroid
FROM calc_centroid
"""
execute_query(sedona, sql_area)


+-------+--------+-----------------+-----------------+
|     id|layer_id|       x_centroid|       y_centroid|
+-------+--------+-----------------+-----------------+
|3662681|     170|37.38866247025377|55.53182179020992|
+-------+--------+-----------------+-----------------+

Функция execute_query выполнена за 0.5045 секунд
+-------+--------+------------------+------------------+
|     id|layer_id|        x_centroid|        y_centroid|
+-------+--------+------------------+------------------+
|3662681|     170| 37.38866247025377| 55.53182179020992|
|3490743|     165| 37.59849062830519|55.686121806286735|
|3490744|     165| 37.59849183075007| 55.68612399634634|
|3492130|     165| 37.55909476376427| 55.72458560632351|
|3490745|     165| 37.61549707775401|55.651902858414715|
|3490746|     165|37.571320209093585| 55.74867654719137|
|3490750|     165| 37.49723819604017| 55.64501241954567|
|3574945|     168|       37.71115234|        55.8256337|
|3490747|     165| 37.61219575487193| 55.7110332

# Переставленные координаты центроида полигона

In [21]:
sql_area = """
    with flipped_cenroid_cord as (
        SELECT id, layer_id, ST_FlipCoordinates(ST_Centroid(geometry)) as centroid FROM public.features_plain LIMIT 100000)
        select id, layer_id, ST_X(centroid) as X_centroid, ST_Y(centroid) as Y_centroid FROM flipped_cenroid_cord"""
execute_query(sedona, sql_area)

+-------+--------+------------+------------+
|     id|layer_id|  x_centroid|  y_centroid|
+-------+--------+------------+------------+
|3581466|     168|55.480520071|37.391561138|
|3581467|     168| 55.76342805| 37.71574541|
|3581468|     168| 55.64437001| 37.58012419|
|3581469|     168|55.520399786|37.218294114|
|3581470|     168|55.882651353|37.535277471|
|3581471|     168| 55.73730632| 37.59808068|
|3581472|     168| 55.74021579| 37.83281142|
|3581473|     168| 55.75639206|37.810272431|
|3581474|     168| 55.79495454| 37.78361788|
|3581475|     168|55.475855384|37.544168038|
+-------+--------+------------+------------+
only showing top 10 rows

Функция execute_query выполнена за 1.0729 секунд


# Центроид полигона в WKT формате с переставленными координатами

In [25]:
sql_area = """
    with flipped_cenroid_cord as (
        select id, layer_id, ST_FlipCoordinates(ST_Centroid(geometry)) as centroid FROM public.features_plain LIMIT 100000)
        select id, layer_id, ST_AsText(centroid) as flipped_centroid FROM flipped_cenroid_cord"""
execute_query(sedona, sql_area)

+-------+--------+--------------------+
|     id|layer_id|    flipped_centroid|
+-------+--------+--------------------+
|3790854|     173|POINT(55.67161239...|
|3790855|     173|POINT(55.66486676...|
|3790856|     173|POINT(55.67252417...|
|3790857|     173|POINT(55.74502475...|
|3790858|     173|POINT(55.65767926...|
|3790859|     173|POINT(55.66453899...|
|3790860|     173|POINT(55.60882024...|
|3790861|     173|POINT(55.61018181...|
|3790862|     173|POINT(55.65784258...|
|3790863|     173|POINT(55.66971484...|
+-------+--------+--------------------+
only showing top 10 rows

Функция execute_query выполнена за 1.3612 секунд


# Геометрия полигона с переставленными координатами в WKT формате

In [27]:
sql_area = """ SELECT id, layer_id, ST_FlipCoordinates(geometry) as flipped_geoemtry FROM public.features_plain LIMIT """
execute_query_all(sedona, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_geoemtry|
+-------+--------+--------------------+
|3382741|     163|0101000020E610000...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.6058 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_geoemtry|
+-------+--------+--------------------+
|3382741|     163|0101000020E610000...|
|3599538|     169|0101000020E610000...|
|3599539|     169|0101000020E610000...|
|3599540|     169|0101000020E610000...|
|3599541|     169|0101000020E610000...|
|3599542|     169|0101000020E610000...|
|3599543|     169|0101000020E610000...|
|3599544|     169|0101000020E610000...|
|3599545|     169|0101000020E610000...|
|3382742|     163|0101000020E610000...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.6196 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id|    flipped_geoemtry|
+-------+--------+

# Ближайшие объекты. 
## Находим ближайшие объекты к данному с помощью KNN

In [8]:
df = sedona.read \
        .format("jdbc") \
        .option("url", postgresql_url) \
        .option("user", credentials.get('user')) \
        .option("password", credentials.get('password')) \
        .option("dbtable", "public.features_plain") \
        .option("driver", "org.postgresql.Driver") \
        .load()
    
# 2. Регистрируем как временную таблицу
df.createOrReplaceTempView("features_plain")

In [9]:
sql_area = """
WITH source_geometry AS (
    SELECT ST_GeomFromWKB(geometry) as geometry
    FROM features_plain
    WHERE id = 3724141 AND layer_id = 170
    LIMIT 1
)
SELECT 
    st.id,
    st.layer_id,
    ST_Distance(so.geometry, ST_GeomFromWKB(st.geometry)) as distance
FROM source_geometry so
CROSS JOIN features_plain st
WHERE NOT (st.id = 3724141 AND st.layer_id = 170)
  AND ST_Distance(so.geometry, ST_GeomFromWKB(st.geometry)) IS NOT NULL
ORDER BY distance
LIMIT 50
"""

credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
postgresql_url = f"jdbc:postgresql://{credentials.get('host')}:{credentials.get('port')}/gisdb_8411_250226?currentSchema=egip"

start = time.time()
result_df = sedona.sql(sql_area)
result_df.show(10)
end = time.time()
print(f"Функция выполнена за {end - start:.4f} секунд")

+-------+--------+--------+
|     id|layer_id|distance|
+-------+--------+--------+
|1973685|     116|     0.0|
|1575008|     100|     0.0|
|1973789|     116|     0.0|
|3781881|     173|     0.0|
|3811204|     173|     0.0|
|3816099|     173|     0.0|
|3778797|     173|     0.0|
|1812227|     111|     0.0|
|1812331|     111|     0.0|
|1575003|     100|     0.0|
+-------+--------+--------+
only showing top 10 rows

Функция выполнена за 64.5410 секунд


# Упрощение геометрии объекта: ST_SimplifyPreserveTopology

In [7]:
sql_area = """ SELECT id, layer_id, ST_SimplifyPreserveTopology(geometry, 10) as simplified_geoemtry FROM public.features_plain LIMIT """
execute_query_all(sedona, sql_area)

1 запись: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+--------------------+
|3518928|     165|0103000020E610000...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.7931 секунд

 10 записей: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+--------------------+
|3518928|     165|0103000020E610000...|
|3518929|     165|0103000020E610000...|
|3518930|     165|0103000020E610000...|
|3518931|     165|0103000020E610000...|
|3518971|     165|0103000020E610000...|
|3518932|     165|0103000020E610000...|
|3518933|     165|0103000020E610000...|
|3518935|     165|0103000020E610000...|
|3518936|     165|0103000020E610000...|
|3518937|     165|0103000020E610000...|
+-------+--------+--------------------+

Функция execute_query выполнена за 0.5765 секунд

 100 записей: 
+-------+--------+--------------------+
|     id|layer_id| simplified_geoemtry|
+-------+--------+

# Определить административную принадлежность объекта к округу и району
## Забираем актуальные границы оркгуов и районов из базы

In [10]:
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
postgresql_url = f"jdbc:postgresql://{credentials.get('host')}:{credentials.get('port')}/mkgh_monitorings"

In [11]:
sql_get_regions = """select * from nsi.nsi_moscow_regions"""
df_regions = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({sql_get_regions}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
df_regions.show(10)

+---+---------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+
| id|region_id|           full_name|                name|short_name|       geometry_4326|       centroid_4326|              x_4326|              y_4326|       geometry_3857|       centroid_3857|              x_3857|              y_3857|         layer_alias|       creation_date|         start_date|           end_date|
+---+---------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+
|  1| 11001200|Троицкий и Новомо...|Троицки

In [12]:
sql_get_districts = """select * from nsi.nsi_moscow_districts"""
df_districts = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({sql_get_districts}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
df_districts.show(10)
df_districts.createOrReplaceTempView("districts_table")

+---+-----------+--------------------+--------------+------------------+---------+--------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+-------------------+
| id|district_id|           full_name|          name|        short_name|region_id|   region_name|region_short_name|       geometry_4326|       centroid_4326|              x_4326|              y_4326|       geometry_3857|       centroid_3857|              x_3857|              y_3857|         layer_alias|       creation_date|         start_date|           end_date|
+---+-----------+--------------------+--------------+------------------+---------+--------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------

In [13]:
credentials = dict({"host": "10.6.81.133", "port":"5432", "user":"postgres", "password":"tpY7H&sdvsdfsdf7zx9J"})
postgresql_url = f"jdbc:postgresql://{credentials.get('host')}:{credentials.get('port')}/gisdb_8411_250226?currentSchema=egip"

sql_get_features_plain = """select * from public.features_plain"""
result_df = sedona.read \
    .format("jdbc") \
    .option("url", postgresql_url) \
    .option("user", f"{credentials.get('user')}") \
    .option("password", f"{credentials.get('password')}") \
    .option("dbtable", f"({sql_get_features_plain}) as subquery") \
    .option("driver", "org.postgresql.Driver") \
    .load()
result_df.createOrReplaceTempView("features_table")

In [14]:
sql_query = """
WITH source_objects AS (
    SELECT 
        id,
        layer_id,
        ST_Transform(ST_MakeValid(ST_SetSRID(ST_GeomFromWKB(geometry), 4326)), 'EPSG:3857') AS geometry_3857 
    FROM features_table
    LIMIT 100000
),
districts_transformed as (
    select  district_id, short_name, region_id, region_short_name,  ST_SetSRID(ST_GeomFromWKB(geometry_3857), 3857) as geometry_3857
    from districts_table
    ),
intersections_with_districts AS (
    SELECT 
        so.id,
        so.layer_id,
        md.district_id,
        md.short_name AS short_district_name,
        md.region_id,
        md.region_short_name AS region_short_name,
        ST_Area(ST_Intersection(so.geometry_3857, md.geometry_3857)) AS area_intersection
    FROM districts_transformed AS md
    CROSS JOIN source_objects AS so
    WHERE ST_Intersects(so.geometry_3857, md.geometry_3857)
),
ranked_intersections AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (PARTITION BY id, layer_id ORDER BY area_intersection DESC) AS rn
    FROM intersections_with_districts
)
SELECT 
    id,
    layer_id,
    district_id,
    short_district_name,
    region_id,
    region_short_name,
    area_intersection
FROM ranked_intersections
WHERE rn = 1
"""
start = time.time()
result_df = sedona.sql(sql_query)
result_df.show(10)
end = time.time()
print(f"Функция  выполнена за {end - start:.4f} секунд")

+-------+--------+-----------+--------------------+---------+-----------------+--------------------+
|     id|layer_id|district_id| short_district_name|region_id|region_short_name|   area_intersection|
+-------+--------+-----------+--------------------+---------+-----------------+--------------------+
|1973560|     116|        203| р-н Бескудниковский|      200|              САО| 1.037705444730578E7|
|1973567|     116|        712|         р-н Ясенево|      700|             ЮЗАО| 7.980473926958346E7|
|1973568|     116|        819|  р-н Фили-Давыдково|      800|              ЗАО| 2.195415403326495E7|
|1973570|     116|        616|р-н Орехово-Борис...|      600|              ЮАО|2.3667924622539833E7|
|1973572|     116|       1212|        р-н Вороново| 11001200|            ТиНАО| 6.355527003152181E8|
|1973573|     116|       1011|          р-н Силино|     1000|            ЗелАО|2.9666502272170234E7|
|1973574|     116|       1112|         р-н Внуково| 11001200|            ТиНАО| 7.956705290